# RakshPulse: PPG Signal and Dataset Exploration

Exploration of the data the RakshPulse model is trained on: the public **Auricle** dataset (healthy cohort, earlobe
PPG from a Nonin transmissive sensor, CNAP continuous blood pressure, two sessions on different days). Everything
uses the production code (`rakshpulse.features`, `rakshpulse.preprocessing`) and the committed tables in
`data/auricle/`; the raw waveform arrays (about 350 MB) are downloaded from the GitHub release if missing.

The modelling itself is in `RakshPulse_Complete_ML.ipynb`. This notebook answers the questions behind it: what does
the signal look like, how much of it is usable, how much does BP vary within and between people, and how strongly do
pulse features relate to BP?

> Earlobe PPG from a research sensor is not MAX30102 behind-the-ear PPG. Section 8 lists the differences.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rakshpulse.ml import DatasetPaths, build_ear_dataset, load_ml_dataset
from rakshpulse.features.feature_pipeline import preprocess_channel
from rakshpulse.preprocessing.segmentation import detect_feet, detect_peaks

pd.set_option("display.width", 160)
paths = DatasetPaths()
windows = pd.read_csv(paths.auricle / "windows.csv", low_memory=False)
subjects = pd.read_csv(paths.auricle / "subjects.csv")
d = load_ml_dataset(paths) if paths.ml_dataset.exists() else build_ear_dataset(paths)[0]
npz = paths.auricle / "waveforms" / "auricle_waveforms_session1.npz"
if not npz.exists():
    from rakshpulse.data.release import fetch
    fetch("waveforms")
wav = np.load(npz, allow_pickle=False)
ids = wav["window_id"]
pos = {w: k for k, w in enumerate(ids)}
print(f"{len(windows):,} windows, {windows.subject_id.nunique()} subjects; model-ready: {len(d):,} windows, "
      f"{d.subject_id.nunique()} subjects")

30,841 windows, 42 subjects; model-ready: 9,851 windows, 40 subjects


## 1. One minute of raw earlobe PPG
Six consecutive non-overlapping 10 s windows of one session-1 recording, with the CNAP reference.

In [2]:
w1 = windows[windows.session == 1].sort_values(["record_id", "t_start_s"])
rec = w1[w1.ear_sqi_ok & w1.label_valid].record_id.value_counts().index[0]
g = w1[(w1.record_id == rec) & (w1.t_start_s % 10 == 0)].iloc[20:26]
sig = np.concatenate([wav["raw_ear"][pos[w]].astype(float) for w in g.window_id])
t = np.arange(sig.size) / 100 + g.t_start_s.iloc[0]
fig, ax = plt.subplots(2, 1, figsize=(13, 5), sharex=True)
ax[0].plot(t, sig, color="#1B2A4A", lw=0.7)
ax[0].set(title=f"Recording {rec}: raw earlobe PPG", ylabel="mV")
ax[1].step(g.t_start_s, g.sbp_mmHg, where="post", color="#C0392B", label="SBP")
ax[1].step(g.t_start_s, g.dbp_mmHg, where="post", color="#0F8B8D", label="DBP")
ax[1].set(title="CNAP reference (window mean)", xlabel="recording time (s)", ylabel="mmHg")
ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()

## 2. One window through the production preprocessing
Detrend, Hampel despiking, 0.5–10 Hz zero-phase band-pass, orientation and z-scoring, then beat segmentation and the quality gate.

In [3]:
raw = wav["raw_ear"][pos[g.window_id.iloc[0]]].astype(float)
pc = preprocess_channel(raw, 100.0, has_dc=False)
peaks = detect_peaks(pc.filtered, 100.0)
feet = detect_feet(pc.filtered, peaks)
fig, ax = plt.subplots(figsize=(13, 3))
ax.plot(np.arange(pc.filtered.size) / 100, pc.filtered, color="#0F8B8D")
ax.plot(peaks / 100, pc.filtered[peaks], "^", color="#C0392B", label="systolic peak")
ax.plot(feet / 100, pc.filtered[feet], "v", color="#D9731A", label="beat onset")
ax.set(title=f"Quality gate: acceptable={pc.sqi.acceptable}, {pc.sqi.n_beats} beats, template r={pc.sqi.template_corr:.2f}",
       xlabel="s", ylabel="z")
ax.legend(frameon=False); plt.show()

## 3. How much of the signal is usable?
Quality-gate pass rate by activity, and the most common rejection reasons.

In [4]:
by_act = windows.groupby("activity_name").agg(windows=("window_id", "size"), ear_pass=("ear_sqi_ok", "mean"),
                                              label_valid=("label_valid", "mean")).sort_values("windows", ascending=False)
display(by_act.round(2))
reasons = windows.loc[~windows.ear_sqi_ok.astype(bool), "ear_sqi_reasons"].fillna("").str.split("|").explode()
print(reasons[reasons != ""].value_counts().head(8))

,windows,ear_pass,label_valid
activity_name,,,
rest,15430,0.80,0.67
not_applicable,5921,0.64,0.04
exercise,2731,0.55,0.58
cold_pressor,2520,0.73,0.67
valsalva,1767,0.77,0.65
breath_hold,1740,0.83,0.67
unlabelled,732,0.74,0.30


ear_sqi_reasons
low_template_correlation    7327
no_signal_flatline          3458
irregular_beat_intervals    2529
too_few_beats               1424
low_cardiac_power           1188
implausible_heart_rate       196
clipped_at_rail              121
Name: count, dtype: int64


## 4. Blood-pressure variation: between people versus within people
A calibration-free model has to explain differences *between* people; a calibrated model only needs changes *within* a person.

In [5]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, t in zip(ax, ("sbp", "dbp")):
    order = d.groupby("subject_id")[t].median().sort_values().index
    a.boxplot([d.loc[d.subject_id == s, t] for s in order], showfliers=False, widths=0.6)
    a.set(title=f"{t.upper()} per subject (sorted by median)", xlabel="subject", ylabel="mmHg", xticks=[])
plt.show()
for t in ("sbp", "dbp"):
    between = d.groupby("subject_id")[t].mean().var()
    within = d.groupby("subject_id")[t].var().mean()
    print(f"{t.upper()}: between-subject variance {between:.0f}, mean within-subject variance {within:.0f} "
          f"-> between-subject share {between / (between + within):.0%}")

SBP: between-subject variance 184, mean within-subject variance 191 -> between-subject share 49%
DBP: between-subject variance 83, mean within-subject variance 79 -> between-subject share 51%


## 5. How strongly do pulse features relate to BP?
Spearman ρ at the subject level (subject medians) and within subjects (pooled ranks after removing each subject's median).

In [6]:
feats = [c for c in d.columns if c.startswith("ear_")]
subj = d.groupby("subject_id")[feats + ["sbp"]].median()
between = subj[feats].corrwith(subj.sbp, method="spearman")
centred = d[feats + ["sbp"]] - d.groupby("subject_id")[feats + ["sbp"]].transform("median")
within = centred[feats].corrwith(centred.sbp, method="spearman")
tab = pd.DataFrame({"between_subjects": between, "within_subjects": within}).dropna()
tab = tab.reindex(tab.between_subjects.abs().sort_values(ascending=False).index)
display(tab.head(12).round(2))
print(f"largest |rho| between subjects: {tab.between_subjects.abs().max():.2f}; within subjects: {tab.within_subjects.abs().max():.2f}")

,between_subjects,within_subjects
ear_w90_n,0.41,0.15
ear_sys_ratio,0.39,0.29
ear_rmssd,-0.38,0.07
ear_w75_n,0.37,0.15
ear_sdnn,-0.35,0.05
ear_h3_h1,-0.35,-0.29
ear_dia_T,-0.34,-0.22
ear_hr,0.32,0.18
ear_beat_T,-0.32,-0.17
ear_win_kurt,-0.31,-0.01


largest |rho| between subjects: 0.41; within subjects: 0.32


## 6. Average pulse shape by SBP tertile

In [7]:
w1v = w1[w1.ear_sqi_ok & w1.label_valid & (w1.activity_name == "rest")]
tert = pd.qcut(w1v.groupby("subject_id").sbp_mmHg.mean(), 3, labels=["low", "middle", "high"])
fig, ax = plt.subplots(figsize=(8, 4))
for grp, c in (("low", "#0F8B8D"), ("middle", "#9AA3AE"), ("high", "#C0392B")):
    beats = []
    for w in w1v[w1v.subject_id.isin(tert[tert == grp].index)].groupby("subject_id").head(10).window_id:
        for b in preprocess_channel(wav["raw_ear"][pos[w]].astype(float), 100.0, has_dc=False).beats:
            beats.append(np.interp(np.linspace(0, len(b) - 1, 100), np.arange(len(b)), (b - b.min()) / np.ptp(b)))
    ax.plot(np.linspace(0, 1, 100), np.mean(beats, axis=0), color=c, lw=2, label=f"{grp} SBP tertile ({len(beats)} beats)")
ax.set(title="Ensemble-average beat (rest, session 1)", xlabel="fraction of beat", ylabel="normalised amplitude")
ax.legend(frameon=False); plt.show()

## 7. Day-to-day shift
The same people on two days: per-subject median heart rate, a shape feature and SBP in session 1 vs session 2.

In [8]:
both = d.groupby(["subject_id", "session"])[["ear_hr", "ear_sys_ratio", "sbp"]].median().unstack("session").dropna()
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, f in zip(ax, ("ear_hr", "ear_sys_ratio", "sbp")):
    x, y = both[(f, 1)], both[(f, 2)]
    a.scatter(x, y, color="#1B2A4A")
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    a.plot([lo, hi], [lo, hi], color="#C0392B", lw=1)
    a.set(title=f"{f}: r = {np.corrcoef(x, y)[0, 1]:.2f}", xlabel="session 1", ylabel="session 2")
plt.tight_layout(); plt.show()
print(f"{len(both)} subjects with both sessions")

32 subjects with both sessions


## 8. What this means for MAX30102

| Property | Auricle training data | RakshPulse target (MAX30102) |
|---|---|---|
| Optics | Transmissive (light passes through the earlobe) | Reflective (LEDs and photodetector on the same face) |
| Site | Earlobe | Behind the ear (posterior auricular artery) |
| Wavelength | Nonin oximeter channel | Red ~660 nm and IR ~880 nm |
| Signal units | Analogue output in mV | 18-bit ADC counts with a large DC level |
| Sampling | 1 kHz original (stored at 100 Hz) | 50–1000 Hz, typically 100 Hz |
| Contact | Clip | Band or adhesive; pressure varies |

The production pipeline removes the DC level and amplitude (band-pass, z-scoring, shape features normalised by beat
duration), so the *code* runs unchanged on MAX30102 counts. Whether the *feature–BP relationships* above transfer is
unknown: the feature distributions will differ, and the model must be retrained on MAX30102 recordings with cuff
references. Until then, MAX30102 outputs are demonstration-only.